### Import required modules

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

import numpy as np

sns.set_style("whitegrid")

## Load all dataset using python

In [2]:
train_df = pd.read_csv("./datasets/train.csv")
val_df = pd.read_csv("./datasets/val.csv")
test_df = pd.read_csv("./datasets/test.csv")

train_df.head()

,Year,dengue_total,Location,Month,monthly_avg_temperature,avg_daily_rain,avg_daily_humidity,avg_daily_soil_moisture,avg_daily_soil_temperature,avg_daily_snowfall,avg_daily_precipitation,daily_rain_density,average_humidity,avg_soil_moisture,avg_soil_temperature,avg_snowfall,avg_precipitation
0,2024,0,KALIKOT,May,10.223880,19.783870,88.951584,0.418761,10.755308,0.000000,19.783870,high,extreme,high,low,none,high
1,2022,0,PARSA,Feb,30.428612,6.673333,62.701965,0.234122,28.814114,0.000000,6.673333,moderate,high,moderate,high,none,moderate
2,2024,19,SALYAN,Sep,10.010685,0.100000,70.479485,0.319492,11.226091,0.000000,0.100000,low,high,high,low,none,low
3,2022,1,BAJHANG,Jan,-6.802581,0.000000,34.566444,0.366226,-1.123817,0.388387,0.551613,no,low,high,low,light,low
4,2022,0,KALIKOT,May,10.058961,13.132258,89.504610,0.415793,10.209129,0.000000,13.132258,high,extreme,high,low,none,high


## Use only selected features for linear regression

In [3]:
selected_columns = [
    # "Year",
    "dengue_total",
    "Location",
    "Month",
    "monthly_avg_temperature",
    "avg_daily_rain",
    "avg_daily_humidity",
    "avg_daily_soil_moisture",
    # "avg_daily_soil_temperature",
    # "avg_daily_snowfall",
    # "avg_daily_precipitation"
]

train = train_df[selected_columns]
val = val_df[selected_columns]
test = test_df[selected_columns]

all_locations = sorted(pd.concat([train["Location"], val["Location"], test["Location"]]).unique())
all_months = sorted(pd.concat([train["Month"], val["Month"], test["Month"]]).unique())

train_loc = pd.get_dummies(train["Location"].astype(pd.CategoricalDtype(all_locations)), dtype=int)
train_month = pd.get_dummies(train["Month"].astype(pd.CategoricalDtype(all_months)), dtype=int)

val_loc = pd.get_dummies(val["Location"].astype(pd.CategoricalDtype(all_locations)), dtype=int)
val_month = pd.get_dummies(val["Month"].astype(pd.CategoricalDtype(all_months)), dtype=int)

test_loc = pd.get_dummies(test["Location"].astype(pd.CategoricalDtype(all_locations)), dtype=int)
test_month = pd.get_dummies(test["Month"].astype(pd.CategoricalDtype(all_months)), dtype=int)

train = pd.concat([train.drop(columns=["Month", "Location"]), train_loc, train_month], axis=1)
val = pd.concat([val.drop(columns=["Month", "Location"]), val_loc, val_month], axis=1)
test = pd.concat([test.drop(columns=["Month", "Location"]), test_loc, test_month], axis=1)

train.head()

,dengue_total,monthly_avg_temperature,avg_daily_rain,avg_daily_humidity,avg_daily_soil_moisture,ACHHAM,ARGHAKHANCHI,BAGLUNG,BAITADI,BAJHANG,...,Dec,Feb,Jan,Jul,Jun,Mar,May,Nov,Oct,Sep
0,0,10.223880,19.783870,88.951584,0.418761,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,0,30.428612,6.673333,62.701965,0.234122,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
2,19,10.010685,0.100000,70.479485,0.319492,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,1,-6.802581,0.000000,34.566444,0.366226,0,0,0,0,1,...,0,0,1,0,0,0,0,0,0,0
4,0,10.058961,13.132258,89.504610,0.415793,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


# Extract X and Y from data

### Normalize and prepare


In [4]:
def normalize_train(x):
    mean = np.nanmean(x, axis=0)
    std = np.nanstd(x, axis=0)
    std = np.where(std == 0, 1, std)
    x_norm = (x - mean) / std
    x_norm = np.nan_to_num(x_norm)
    return x_norm, mean, std

def normalize_pred(x, mean, std):
    std = np.where(std == 0, 1, std)
    x_norm = (x - mean) / std
    return np.nan_to_num(x_norm)


Training set

In [5]:
x_train= train.drop(columns=["dengue_total"])
# all the features
features = x_train.columns.to_list()

x_train, x_mean, x_std = normalize_train(x_train.to_numpy())
y_train, y_mean, y_std = normalize_train(train["dengue_total"].to_numpy())

print("Columns of features:")
print(features)

print(f"Shape of x: {x_train.shape}")

Columns of features:
['monthly_avg_temperature', 'avg_daily_rain', 'avg_daily_humidity', 'avg_daily_soil_moisture', 'ACHHAM', 'ARGHAKHANCHI', 'BAGLUNG', 'BAITADI', 'BAJHANG', 'BAJURA', 'BANKE', 'BARA', 'BARDIYA', 'BHAKTAPUR', 'BHOJPUR', 'DADELDHURA', 'DAILEKH', 'DANG', 'DARCHULA', 'DHADING', 'DHANKUTA', 'DHANUSA', 'DOLAKHA', 'DOLPA', 'DOTI', 'GORKHA', 'GULMI', 'HUMLA', 'ILAM', 'JAJARKOT', 'JHAPA', 'JUMLA', 'KAILALI', 'KALIKOT', 'KANCHANPUR', 'KASKI', 'KATHMANDU', 'KHOTANG', 'LALITPUR', 'LAMJUNG', 'MAHOTTARI', 'MAKWANPUR', 'MANANG', 'MORANG', 'MUGU', 'MUSTANG', 'MYAGDI', 'NUWAKOT', 'OKHALDHUNGA', 'PALPA', 'PANCHTHAR', 'PARBAT', 'PARSA', 'PYUTHAN', 'RAMECHHAP', 'RASUWA', 'RAUTAHAT', 'ROLPA', 'RUPANDEHI', 'SALYAN', 'SANKHUWASABHA', 'SAPTARI', 'SARLAHI', 'SINDHULI', 'SINDHUPALCHOK', 'SIRAHA', 'SOLUKHUMBU', 'SUNSARI', 'SURKHET', 'SYANGJA', 'TANAHU', 'TAPLEJUNG', 'UDAYAPUR', 'Apr', 'Aug', 'Dec', 'Feb', 'Jan', 'Jul', 'Jun', 'Mar', 'May', 'Nov', 'Oct', 'Sep']
Shape of x: (62596, 85)


Validation and Test set

In [6]:
# Validation Set
x_val = normalize_pred(val.drop(columns=["dengue_total"]).to_numpy(), x_mean, x_std)
y_val = normalize_pred(val["dengue_total"].to_numpy(), y_mean, y_std)

# Testing Set
x_test = normalize_pred(test.drop(columns=["dengue_total"]).to_numpy(), x_mean, x_std)
y_test = normalize_pred(test["dengue_total"].to_numpy(),y_mean, y_std)

print(f"Validation and test sets are ready to use!")

Validation and test sets are ready to use!


# LinearRegression Architecture

In [7]:
class LinearRegression:
    def __init__(self, shape, lr=0.01):
        self.w = np.random.randn(shape)
        self.b = 0.0
        self.lr = lr

    def fit(self, x, y):
        y_pred = x.dot(self.w) + self.b
        error = y_pred - y

        grad_w = x.T.dot(error) / len(y)
        grad_b = np.mean(error)

        self.w -= self.lr * grad_w
        self.b -= self.lr * grad_b

        return np.mean(error ** 2)

    def predict(self, x):
        return x.dot(self.w) + self.b


# Training Linear Model in 100 Epochs

In [8]:
model = LinearRegression(x_train.shape[1], 0.01)
train_losses = []
val_losses = []

In [12]:
for epoch in range(1, 501):
    train_loss = model.fit(x=x_train, y=y_train)
    train_losses.append(train_loss)

    preds = model.predict(x_val)
    val_loss = np.mean((preds - y_val) ** 2)
    val_losses.append(val_loss)

    if epoch % 5 == 0:
        print(f"Epoch {epoch}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")


Epoch 5: Train Loss=0.8702, Val Loss=1.1520
Epoch 10: Train Loss=0.8702, Val Loss=1.1520
Epoch 15: Train Loss=0.8702, Val Loss=1.1520
Epoch 20: Train Loss=0.8702, Val Loss=1.1520
Epoch 25: Train Loss=0.8702, Val Loss=1.1520
Epoch 30: Train Loss=0.8702, Val Loss=1.1520
Epoch 35: Train Loss=0.8702, Val Loss=1.1520
Epoch 40: Train Loss=0.8702, Val Loss=1.1519
Epoch 45: Train Loss=0.8702, Val Loss=1.1519
Epoch 50: Train Loss=0.8701, Val Loss=1.1519
Epoch 55: Train Loss=0.8701, Val Loss=1.1519
Epoch 60: Train Loss=0.8701, Val Loss=1.1519
Epoch 65: Train Loss=0.8701, Val Loss=1.1519
Epoch 70: Train Loss=0.8701, Val Loss=1.1519
Epoch 75: Train Loss=0.8701, Val Loss=1.1519
Epoch 80: Train Loss=0.8701, Val Loss=1.1519
Epoch 85: Train Loss=0.8701, Val Loss=1.1519
Epoch 90: Train Loss=0.8701, Val Loss=1.1519
Epoch 95: Train Loss=0.8701, Val Loss=1.1519
Epoch 100: Train Loss=0.8701, Val Loss=1.1519
Epoch 105: Train Loss=0.8701, Val Loss=1.1519
Epoch 110: Train Loss=0.8701, Val Loss=1.1519
Epoch 11